# Data Preprocessing Pipeline
This notebook processes the raw extracted river level data into a training-ready dataset based on instructions in `preprocessing.md`.


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


### 1. Load Data


In [ ]:
df = pd.read_csv('extracted_data_new copy 3.csv')
print(f"Original shape: {df.shape}")
df.head()


### 1.1 Fix Data Types & 1.4 Drop Useless Column
Convert everything to correct types, drop completely empty columns.


In [ ]:
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
numeric_cols = ['water_level', 'previous_water_level', 'rainfall_mm', 'minor_flood_level', 'major_flood_level']
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

if 'trend' in df.columns:
    df = df.drop(columns=['trend'])


### 1.2 Remove Duplicates & 1.3 Sort Data
Group by station and datetime, keeping the latest entry. Sort chronologically per station.


In [ ]:
df = df.drop_duplicates(subset=['station', 'datetime'], keep='last')
df = df.sort_values(by=['station', 'datetime']).reset_index(drop=True)
print(f"Shape after deduplication: {df.shape}")


### 4.6 Missing Indicators (Before Imputation)
Create flags for inherently missing data to help the model quantify uncertainty.


In [ ]:
df['water_level_was_missing'] = df['water_level'].isna().astype(int)
df['rainfall_was_missing'] = df['rainfall_mm'].isna().astype(int)


### 2. Missing Values Handling
Fill categorical with explicit 'Unknown'/'Mode'. Fill small numerical gaps with interpolation. Leave massive 2025 Jan-Jun gaps untouched as nan sequences.


In [ ]:
# Categorical
df['rainfall_type'] = df['rainfall_type'].fillna('Unknown')
df['status'] = df['status'].fillna('Unknown')
df['river_basin'] = df.groupby('station')['river_basin'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Unknown'))

# Numeric: Small Gaps Only (limit=2 items, typical for ~6 hour interpolation if interval=3h)
df['water_level'] = df.groupby('station')['water_level'].transform(lambda x: x.interpolate(limit=2))
df['previous_water_level'] = df.groupby('station')['previous_water_level'].transform(lambda x: x.interpolate(limit=2))

# Rainfall missing -> Assume 0
df['rainfall_mm'] = df['rainfall_mm'].fillna(0)

# Flood Thresholds -> Consistent Forward/Backward Fill
df['minor_flood_level'] = df.groupby('station')['minor_flood_level'].transform(lambda x: x.ffill().bfill())
df['major_flood_level'] = df.groupby('station')['major_flood_level'].transform(lambda x: x.ffill().bfill())


### 3. Time Consistency
Measure time jumps backwards to flag missing interval epochs.


In [ ]:
df['time_diff_hours'] = df.groupby('station')['datetime'].diff().dt.total_seconds() / 3600.0
df['gap_flag_6h'] = (df['time_diff_hours'] > 6).astype(int)
df['gap_flag_24h'] = (df['time_diff_hours'] > 24).astype(int)


### 4.1 & 4.2 Time & Seasonal Features
Extract cyclic temporal attributes.


In [ ]:
df['hour'] = df['datetime'].dt.hour
df['day'] = df['datetime'].dt.day
df['month'] = df['datetime'].dt.month
df['day_of_week'] = df['datetime'].dt.dayofweek
df['quarter'] = df['datetime'].dt.quarter
df['is_monsoon'] = df['month'].isin([5, 6, 7, 10, 11, 12]).astype(int)


### 4.3 Lag Features
Compute historical steps. If time breaks occur in between history steps, explicitly mask them as NaNs.


In [ ]:
def create_lags(group):
    # Lags for water_level: 1, 2, 4, 8 steps
    for lag in [1, 2, 4, 8]:
        group[f'water_level_lag_{lag}'] = group['water_level'].shift(lag)
    # Lags for rainfall
    for lag in [1, 2]:
        group[f'rainfall_lag_{lag}'] = group['rainfall_mm'].shift(lag)
    return group

df = df.groupby('station', group_keys=False).apply(create_lags)

# Invalidate lag features spanning a time gap (> 6h)
for lag in [1, 2, 4, 8]:
    # Rolling max of the gap flag up to 'lag' periods behind
    invalid_mask = df.groupby('station')['gap_flag_6h'].rolling(lag, min_periods=1).max().reset_index(level=0, drop=True) > 0
    df.loc[invalid_mask, f'water_level_lag_{lag}'] = np.nan
    if lag in [1, 2]:
        df.loc[invalid_mask, f'rainfall_lag_{lag}'] = np.nan


### 4.4 Rolling Features
Windowed aggregate features over preceding epochs.


In [ ]:
def create_rolling(group):
    group['water_level_roll_mean_3'] = group['water_level'].rolling(3, min_periods=1).mean()
    group['water_level_roll_max_3'] = group['water_level'].rolling(3, min_periods=1).max()
    group['rainfall_roll_sum_3'] = group['rainfall_mm'].rolling(3, min_periods=1).sum()
    return group

df = df.groupby('station', group_keys=False).apply(create_rolling)


### 4.5 Flood Threshold Features
Percentages of capacity against current level.


In [ ]:
df['flood_ratio_minor'] = df['water_level'] / df['minor_flood_level']
df['flood_ratio_major'] = df['water_level'] / df['major_flood_level']
df['above_minor_flag'] = (df['water_level'] > df['minor_flood_level']).astype(int)
df['above_major_flag'] = (df['water_level'] > df['major_flood_level']).astype(int)


### 5. Target Creation
Target defines output predictions. We use Water Level T+1 (next sequence record).


In [ ]:
df['target_water_level_next'] = df.groupby('station')['water_level'].shift(-1)


### 6. Final Cleaning
Drop non-constructible contexts: Unpredictable records lacking Target and fundamental input Lags.


In [ ]:
df_clean = df.dropna(subset=[
    'target_water_level_next', 
    'water_level_lag_1', 
    'water_level_lag_2'
])
# Clean sequences: Don't train on instances directly following a long blank period.
df_clean = df_clean[df_clean['gap_flag_6h'] == 0]
print(f"Shape after strict cleaning: {df_clean.shape}")


### 7. Categorical Encoding
Encode Strings for Tree compatibility (LightGBM/XGBoost).


In [ ]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = ['station', 'river_basin', 'rainfall_type', 'status']

for col in categorical_cols:
    if col in df_clean.columns:
        le = LabelEncoder()
        df_clean[col + '_encoded'] = le.fit_transform(df_clean[col].astype(str))

output_file = 'processed_training_data.csv'
df_clean.to_csv(output_file, index=False)
print(f"Successfully saved cleaned and feature engineered data to {output_file}")
df_clean.head()
